In [1]:
# 读取TOML文件
from pathlib import Path
import tomllib

toml_path = Path("all_tahoe.toml")  # 改成你的路径

with toml_path.open("rb") as f:
    cfg = tomllib.load(f)
    
cfg["fewshot"].keys()

dict_keys(['tahoe.CVCL_1097', 'tahoe.CVCL_1285', 'tahoe.CVCL_1098', 'tahoe.CVCL_0334', 'tahoe.CVCL_0480'])

In [2]:
from itertools import product
val_cartesian_fewshot_list = []
test_cartesian_fewshot_list = []
for x in cfg["fewshot"].keys():
    cell_name = x.split(".")[-1]
    val_drug_names = cfg["fewshot"][x]["val"]
    val_cartesian_fewshot = list(product([cell_name], val_drug_names))
    test_drug_names = cfg["fewshot"][x]["test"]
    test_cartesian_fewshot = list(product([cell_name], test_drug_names))
    val_cartesian_fewshot_list.extend(val_cartesian_fewshot)
    test_cartesian_fewshot_list.extend(test_cartesian_fewshot)

print(val_cartesian_fewshot_list[:10])
print(test_cartesian_fewshot_list[-10:])
print(len(val_cartesian_fewshot_list))
print(len(test_cartesian_fewshot_list))

[('CVCL_1097', "[('Allopurinol', 0.05, 'uM')]"), ('CVCL_1097', "[('Allopurinol', 0.5, 'uM')]"), ('CVCL_1097', "[('Quinestrol', 0.05, 'uM')]"), ('CVCL_1097', "[('Quinestrol', 0.5, 'uM')]"), ('CVCL_1097', "[('Quinestrol', 5.0, 'uM')]"), ('CVCL_1097', "[('Allantoin', 5.0, 'uM')]"), ('CVCL_1097', "[('Allantoin', 0.05, 'uM')]"), ('CVCL_1097', "[('Allantoin', 0.5, 'uM')]"), ('CVCL_1097', "[('Vandetanib', 0.5, 'uM')]"), ('CVCL_1097', "[('Vandetanib', 0.05, 'uM')]")]
[('CVCL_0480', "[('Bestatin', 0.05, 'uM')]"), ('CVCL_0480', "[('Bestatin (hydrochloride)', 0.5, 'uM')]"), ('CVCL_0480', "[('Bestatin', 0.5, 'uM')]"), ('CVCL_0480', "[('Bestatin', 5.0, 'uM')]"), ('CVCL_0480', "[('Dimethyl fumarate', 0.05, 'uM')]"), ('CVCL_0480', "[('Dimethyl fumarate', 0.5, 'uM')]"), ('CVCL_0480', "[('Dimethyl fumarate', 5.0, 'uM')]"), ('CVCL_0480', "[('Posaconazole', 0.05, 'uM')]"), ('CVCL_0480', "[('Posaconazole', 0.5, 'uM')]"), ('CVCL_0480', "[('Posaconazole', 5.0, 'uM')]")]
280
3889


In [ ]:
import pickle
with open("pert_sample_list.pkl", "rb") as f:
    pert_data = pickle.load(f)
    
from tqdm import tqdm
train_sample_list = []
val_sample_list = []
test_sample_list = []
for x in tqdm(pert_data):
    cell_drug_product = (x["cartesian_key"][1], x["cartesian_key"][2])
    if cell_drug_product in val_cartesian_fewshot_list:
        val_sample_list.append(x)
    elif cell_drug_product in test_cartesian_fewshot_list:
        test_sample_list.append(x)
    else:
        train_sample_list.append(x)

In [ ]:
# 分别写入lmdb
import lmdb
import pickle
from tqdm import tqdm

def dump_list_to_lmdb(
    data_list,
    lmdb_path: str,
    map_size_bytes: int,   # 必须足够大（比如原始数据大小 * 1.2~2）
    protocol: int = pickle.HIGHEST_PROTOCOL,
):
    env = lmdb.open(
        lmdb_path,
        map_size=map_size_bytes,
        subdir=True,
        create=True,
        lock=True,
        readahead=False,
        max_dbs=1,
    )

    with env.begin(write=True) as txn:
        # 存长度
        txn.put(b"__len__", str(len(data_list)).encode("utf-8"))

    # 分批写：避免一个 txn 太大
    batch = 10_000
    for start in tqdm(range(0, len(data_list), batch), desc="writing lmdb"):
        end = min(start + batch, len(data_list))
        with env.begin(write=True) as txn:
            for i in range(start, end):
                k = str(i).encode("utf-8")
                v = pickle.dumps(data_list[i], protocol=protocol)
                txn.put(k, v)

    env.sync()
    env.close()

lmdb_dict = {"tahoe_train": train_sample_list, "tahoe_val": val_sample_list, "tahoe_test": test_sample_list}
for key, data_list in lmdb_dict.items():
    dump_list_to_lmdb(data_list, key, map_size_bytes=180 * (1 << 30))
    
    


In [ ]:
import pickle
with open("control_dict.pkl", "rb") as f:
    control_data = pickle.load(f)
    
    
# 字符串，tuple互相转换脚本
import ast
test_tuple = list(control_data.keys())[0]
test_str = str(test_tuple)
print(len(test_tuple))
print(len(test_str))
print(len(ast.literal_eval(test_str)))

def dump_dict_to_lmdb(
    data_dict,
    lmdb_path: str,
    map_size_bytes: int,
    protocol: int = pickle.HIGHEST_PROTOCOL,
):

    env = lmdb.open(
        lmdb_path,
        map_size=map_size_bytes,
        subdir=True,
        create=True,
        lock=True,
        readahead=False,
        max_dbs=1,
    )

    keys = list(data_dict.keys())

    with env.begin(write=True) as txn:
        txn.put(b"__len__", str(len(keys)).encode("utf-8"))
        txn.put(b"__keys__", pickle.dumps(keys, protocol=protocol))

        for k_str, obj in tqdm(data_dict.items()):
            k = str(k_str).encode("utf-8")
            v = pickle.dumps(obj, protocol=protocol)
            txn.put(k, v)

    env.sync()
    env.close()

# 写入lmdb
dump_dict_to_lmdb(control_data, "tahoe_control", map_size_bytes=180 * (1 << 30))

In [13]:
# read_lmdb_test
import pickle
import lmdb
pert_lmdb_path = "/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/master/nemo_cellflow/preprocessing/tahoe/group/tahoe_train"
control_lmdb_path = "/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/master/nemo_cellflow/preprocessing/tahoe/group/tahoe_control"
control_pert = "[('DMSO_TF', 0.0, 'uM')]"
pert_env = lmdb.open(pert_lmdb_path, readonly=True)
control_env = lmdb.open(control_lmdb_path, readonly=True)

with pert_env.begin(write=False) as pert_txn:
    pert_buf = pert_txn.get(str(125).encode("utf-8"))
    pert_group = pickle.loads(pert_buf)
    pert_cartesian_key = pert_group["cartesian_key"]
    control_cartesian_key = (pert_cartesian_key[0], pert_cartesian_key[1], control_pert)
    print(int(pert_txn.get(b"__len__")))
pert_group

120297


{'cartesian_key': ('plate2', 'CVCL_1550', "[('ML264', 0.5, 'uM')]"),
 'cell_matrix': <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 146453 stored elements and shape (1024, 2000)>}

In [14]:
with control_env.begin(write=False) as control_txn:
    control_buf = control_txn.get(str(control_cartesian_key).encode())
    control_group = pickle.loads(control_buf)
    print(int(control_txn.get(b"__len__")))
control_group



700


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 987030 stored elements and shape (5754, 2000)>